# 09 — Segment-Level Analysis and Heterogeneity (Light Touch)
**Prerequisites:** `07_multiple_testing.ipynb` (correcting for many segment cuts);
`01_ab_testing_fundamentals.ipynb`.
**Connects to:** a **later causal-ML module** in this repo will cover heterogeneous treatment
effects (CATE estimators, causal forests, meta-learners) in full mathematical depth — this
notebook stays deliberately light and product-oriented; do not expect HTE/CATE math here.

## Narrative thread
```
Why look at segments -> Simpson's paradox risk (numeric illustration)
   -> multiple-testing correction for segment cuts -> pre-registered vs exploratory segments
```

## Why segment the results

An average treatment effect can mask important differences across subgroups: a feature might
help power users and hurt new users, or help mobile and hurt desktop. Segment-level analysis
answers "for whom does this work," which matters both for rollout decisions (ship to a subset)
and for understanding mechanism. But segment analysis is also one of the easiest ways to fool
yourself, for two distinct reasons covered below: **Simpson's paradox** and **multiple testing**.

## Simpson's paradox risk

Aggregating (or disaggregating) data across a confounding grouping variable can **reverse the
sign** of an apparent effect. In an experiment this typically arises when the mix of segments
differs between arms even though the *within-segment* randomization is correct — e.g. one arm
disproportionately serves a segment that behaves very differently on the metric.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [2]:
# ── Numeric illustration of Simpson's paradox in an A/B test readout ────
# Two segments (new vs. returning users) with different conversion baselines and different
# effective sample splits between arms -- the AGGREGATE comparison can look backwards.
segments = pd.DataFrame({
    'segment':       ['new_user', 'new_user', 'returning_user', 'returning_user'],
    'arm':           ['control',  'treatment', 'control',        'treatment'],
    'n':             [9000,       1000,        1000,             9000],
    'conversions':   [900,        120,         150,              1000],
})
segments['rate'] = segments['conversions'] / segments['n']
print(segments)

# Within each segment, treatment is BETTER:
for seg in segments['segment'].unique():
    sub = segments[segments['segment'] == seg]
    c = sub[sub.arm == 'control']['rate'].values[0]
    t = sub[sub.arm == 'treatment']['rate'].values[0]
    print(f"{seg}: control={c:.3f}, treatment={t:.3f}, treatment {'better' if t>c else 'worse'} within segment")

# But the AGGREGATE (pooled) rates can look different because the arms have very different segment MIXES:
agg = segments.groupby('arm').apply(lambda d: d['conversions'].sum() / d['n'].sum(), include_groups=False)
print("\nAggregate (pooled) conversion rate by arm, ignoring segment mix:")
print(agg)
print("\nThis is why a healthy experiment should have BALANCED segment composition across arms")
print("(guaranteed by correct randomization) -- if it doesn't, investigate SRM (notebook 06) first;")
print("Simpson's paradox in a *properly randomized* experiment is a warning to always check whether")
print("apparent aggregate effects are driven by one dominant segment, not a universal effect.")

          segment        arm     n  conversions      rate
0        new_user    control  9000          900  0.100000
1        new_user  treatment  1000          120  0.120000
2  returning_user    control  1000          150  0.150000
3  returning_user  treatment  9000         1000  0.111111
new_user: control=0.100, treatment=0.120, treatment better within segment
returning_user: control=0.150, treatment=0.111, treatment worse within segment

Aggregate (pooled) conversion rate by arm, ignoring segment mix:
arm
control      0.105
treatment    0.112
dtype: float64

This is why a healthy experiment should have BALANCED segment composition across arms
(guaranteed by correct randomization) -- if it doesn't, investigate SRM (notebook 06) first;
Simpson's paradox in a *properly randomized* experiment is a warning to always check whether
apparent aggregate effects are driven by one dominant segment, not a universal effect.


## Multiple-testing correction for segment cuts

Cutting an experiment's results by many segments (country x device x new/returning x ...) is
exactly the "many tests" scenario from `07_multiple_testing.ipynb` — with dozens of segment
combinations, spurious "this segment shows a huge effect" findings are expected by chance alone.
Apply the same FDR (Benjamini-Hochberg) correction there, or restrict formal inference to a small
number of **pre-registered** segments decided before the experiment ran.

## Pre-registered vs. exploratory segments

| | Pre-registered segments | Exploratory segments |
|---|---|---|
| When decided | Before the experiment starts, based on a specific hypothesis (e.g. "we expect this to matter more for mobile users") | After seeing aggregate results, scanning for "what looks interesting" |
| Statistical treatment | Can use close to nominal alpha, similar to a primary metric | Must be FDR-corrected (notebook 07) and treated as hypothesis-*generating*, not confirmatory |
| Action on a "finding" | Can inform the ship decision directly | Should be validated in a dedicated follow-up experiment before acting on it |

## A note on where this goes next

This notebook deliberately keeps heterogeneity analysis at the level of "cut by known segments,
correct for multiplicity, don't over-interpret." Formal **heterogeneous treatment effect (HTE)**
estimation — conditional average treatment effects (CATE), causal forests, meta-learners (S/T/X-
learners), and doubly robust estimation of who benefits most — is picked up in depth in a
**later causal-ML module** of this repository. Nothing here should be read as a substitute for
that treatment.

## Key takeaways

| Concept | Statement |
|---|---|
| Segment analysis | Useful for "for whom does this work" but easy to over-interpret |
| Simpson's paradox | Aggregate and within-segment effects can point in different directions if segment mix differs across arms |
| Multiplicity | Many segment cuts = many tests; correct with FDR (notebook 07) or pre-register a small segment set |
| Forward pointer | Full CATE/HTE machinery is covered in the later causal-ML module, not here |

## References

- Kohavi, R., Tang, D., & Xu, Y. (2020). *Trustworthy Online Controlled Experiments*, Ch. 20 (segments, heterogeneity, and interpretation pitfalls).
- Simpson, E. H. (1951). The Interpretation of Interaction in Contingency Tables. *JRSS-B*.
- See `07_multiple_testing.ipynb` for the FDR correction to apply across segment cuts.
